# So sánh 3 độ phân giải 96³ / 128³ / 200³ cho model sweep (classification)

Tải 3 dataset consolidated từ HF (private/public) và so sánh trực tiếp, lấy **200³ làm baseline**:

- `tqhuyen/harvard-oct-glaucoma-200` (baseline, raw)
- `tqhuyen/harvard-oct-glaucoma-128`
- `tqhuyen/harvard-oct-glaucoma-96`

4 thí nghiệm: Ảnh (PSNR/SSIM vs 200) · Đặc trưng (CKA/Silhouette vs 200) · Nhiệm vụ (AUC/F1/ECE) · Batch-size.
Kết luận: **bản nhỏ nhất đạt ngưỡng so với 200** = phù hợp nhất cho sweep (nhanh mà không mất chất lượng).

Chạy trên **Colab GPU (L4/A100/H100)**. Lưu TẤT CẢ vào Drive
`MasterBKDN/Thesis/resolution_3way_96_128_200[_figures]`.

## 1. Setup

In [3]:
!git clone --depth 1 https://github.com/Tqhuyen/glaucoma-thesis.git /content/glaucoma-thesis 2>/dev/null || git -C /content/glaucoma-thesis pull --ff-only 2>/dev/null || true
%cd /content/glaucoma-thesis
!pip install -q wandb scikit-learn scipy matplotlib huggingface_hub hf_transfer pandas
!nvidia-smi --query-gpu=name,memory.total --format=csv


Already up to date.
/content/glaucoma-thesis
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [4]:
import os, sys, json, time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, "scripts")
import resolution_study as rs
import compare_resolutions as cr

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Cần GPU: Runtime -> Change runtime type -> L4/A100/H100")

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        print("no HF_TOKEN:", e)
if not HF_TOKEN:
    raise RuntimeError("Thiếu HF_TOKEN (Colab Secrets).")
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login, snapshot_download
login(token=HF_TOKEN)
print("HF login OK")


torch 2.11.0+cu128 | cuda: True NVIDIA A100-SXM4-40GB


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login OK


## 2. Cấu hình

- `BASELINE_RES = 200` là mốc so sánh; `RES_LIST = (128, 96)` là 2 ứng viên cần đánh giá.
- `TRAIN_N/VAL_N = 0` nghĩa là dùng toàn bộ; đặt số nhỏ để chạy nhanh (200³ nặng ~2.4× 128³).

In [5]:
SEED = 42
BASELINE_RES = 200
RES_LIST = (128, 96)
SIZES = (BASELINE_RES,) + tuple(RES_LIST)
HF_REPOS = {
    200: "tqhuyen/harvard-oct-glaucoma-200",
    128: "tqhuyen/harvard-oct-glaucoma-128",
    96: "tqhuyen/harvard-oct-glaucoma-96",
}
DATA_CACHE = "/content/hf_datasets"
SPLIT = "Training"
VAL_SPLIT = "Validation"

TRAIN_N = 0
VAL_N = 0
EPOCHS = 30
BS = 4
BS_LARGE = 8
WIDTH = 24

RUN_EXP1 = True
RUN_EXP2 = True
RUN_EXP3 = True
RUN_EXP4 = True

np.random.seed(SEED)
torch.manual_seed(SEED)
FIG_DIR = os.path.join("figures", "resolution_3way")
os.makedirs(FIG_DIR, exist_ok=True)
DRIVE_ROOT = os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/MasterBKDN/Thesis")
DRIVE_FIG = os.path.join(DRIVE_ROOT, "resolution_3way_96_128_200_figures")
DRIVE_MODEL = os.path.join(DRIVE_ROOT, "resolution_3way_96_128_200")
print("sizes:", SIZES, "| baseline:", BASELINE_RES, "| epochs", EPOCHS, "| bs", BS, BS_LARGE)


sizes: (200, 128, 96) | baseline: 200 | epochs 30 | bs 4 8


## 3. Tải 3 dataset từ HF + nạp memmap

In [6]:
DATA = {}
for s in SIZES:
    repo = HF_REPOS[s]
    local = os.path.join(DATA_CACHE, f"glaucoma_all_{s}")
    print(f"[hf] snapshot {repo} -> {local}", flush=True)
    snapshot_download(repo_id=repo, repo_type="dataset", local_dir=local)
    DATA[s] = {
        "train_v": np.load(os.path.join(local, f"{SPLIT}_volumes.npy"), mmap_mode="r"),
        "train_l": np.load(os.path.join(local, f"{SPLIT}_labels.npy")),
        "val_v": np.load(os.path.join(local, f"{VAL_SPLIT}_volumes.npy"), mmap_mode="r"),
        "val_l": np.load(os.path.join(local, f"{VAL_SPLIT}_labels.npy")),
    }
    print(f"[data] {s}^3 train {DATA[s]['train_v'].shape} val {DATA[s]['val_v'].shape}", flush=True)


[hf] snapshot tqhuyen/harvard-oct-glaucoma-200 -> /content/hf_datasets/glaucoma_all_200


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

[data] 200^3 train (2100, 1, 200, 200, 200) val (300, 1, 200, 200, 200)
[hf] snapshot tqhuyen/harvard-oct-glaucoma-128 -> /content/hf_datasets/glaucoma_all_128


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

[data] 128^3 train (2100, 1, 128, 128, 128) val (300, 1, 128, 128, 128)
[hf] snapshot tqhuyen/harvard-oct-glaucoma-96 -> /content/hf_datasets/glaucoma_all_96


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

[data] 96^3 train (2100, 1, 96, 96, 96) val (300, 1, 96, 96, 96)


## 4. Wandb + Drive helpers

In [8]:
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception as e:
        print("no WANDB_API_KEY:", e)

RUN_NAME = "resolution_3way_96_128_200_" + time.strftime("%Y%m%d_%H%M%S")
run = None
if os.environ.get("WANDB_API_KEY"):
    try:
        import wandb
        run = wandb.init(project="glaucoma-thesis", name=RUN_NAME,
                         config={"sizes": list(SIZES), "baseline": BASELINE_RES,
                                 "epochs": EPOCHS, "bs": BS, "bs_large": BS_LARGE})
    except Exception as e:
        print("[wandb] init failed:", e)
RUN_WANDB = run is not None
print("wandb:", RUN_NAME if RUN_WANDB else None)


def mount_drive():
    if os.path.isdir(DRIVE_ROOT):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return os.path.isdir(DRIVE_ROOT)
    except Exception as e:
        print("[drive] mount skipped:", e)
        return False


DRIVE_READY = mount_drive()
print("drive:", DRIVE_READY, DRIVE_ROOT)


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: huyenquangtran2002 (quang-huyen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: resolution_3way_96_128_200_20260910_080900
drive: True /content/drive/MyDrive/MasterBKDN/Thesis


## 5. Thí nghiệm 1 — Reconstruction vs 200³ (baseline)

Hạ 200³ → {128,96}³ (Gaussian+trilinear) rồi nâng ngược về 200³, đo PSNR/SSIM so với gốc.

In [9]:
exp1 = {}
if RUN_EXP1:
    rv = DATA[BASELINE_RES]["val_v"]
    n = min(50, len(rv))
    idx = np.random.default_rng(0).choice(len(rv), size=n, replace=False)
    for res in RES_LIST:
        ps, ss = [], []
        for j in idx:
            x = np.asarray(rv[j])
            x = (x[0] if x.ndim == 4 else x).astype(np.float32)
            up = rs.resize_volume(rs.downsample_volume(x, (res,) * 3, "gaussian_trilinear"), (BASELINE_RES,) * 3)
            ps.append(rs.psnr(x, up)); ss.append(rs.ssim3d(x, up))
        exp1[res] = {"psnr": float(np.mean(ps)), "ssim": float(np.mean(ss))}
        print(f"[exp1] 200->{res}->200: PSNR {exp1[res]['psnr']:.2f} dB | SSIM {exp1[res]['ssim']:.4f}")
    v = np.asarray(rv[idx[0]]); v = (v[0] if v.ndim == 4 else v).astype(np.float32)
    mid = BASELINE_RES // 2
    panels = [("200^3 (baseline)", v[:, :, mid])]
    for res in RES_LIST:
        up = rs.resize_volume(rs.downsample_volume(v, (res,) * 3, "gaussian_trilinear"), (BASELINE_RES,) * 3)
        panels.append((f"200->{res}->200", up[:, :, mid]))
    fig, axs = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.6))
    for ax, (t, im) in zip(np.atleast_1d(axs), panels):
        ax.imshow(im, cmap="gray", vmin=np.percentile(v, 1), vmax=np.percentile(v, 99)); ax.set_title(t); ax.axis("off")
    p1 = os.path.join(FIG_DIR, "exp1_reconstruction_vs_200.png")
    fig.tight_layout(); fig.savefig(p1, dpi=160, bbox_inches="tight"); plt.close(fig)
    print("wrote", p1)
    if RUN_WANDB:
        import wandb
        for res, d in exp1.items():
            run.log({f"exp1/{res}/psnr": d["psnr"], f"exp1/{res}/ssim": d["ssim"]}, step=1)
        run.log({"exp1/img": wandb.Image(p1)}, step=1)


[exp1] 200->128->200: PSNR 25.63 dB | SSIM 0.3949
[exp1] 200->96->200: PSNR 25.55 dB | SSIM 0.3864
wrote figures/resolution_3way/exp1_reconstruction_vs_200.png


## 6. Thí nghiệm 2 — Proxy Model Training (200 / 128 / 96, cùng seed/hp)

In [10]:
exp2, models, hists = {}, {}, {}
if RUN_EXP2:
    for s in SIZES:
        print(f"[exp2] train {s}^3 ...", flush=True)
        model, met, feats, yv, hist = cr.train_proxy(
            DATA[s]["train_v"], DATA[s]["train_l"], DATA[s]["val_v"], DATA[s]["val_l"],
            BS, EPOCHS, DEVICE, SEED, width=WIDTH, n_train=TRAIN_N, n_val=VAL_N)
        exp2[s], models[s], hists[s] = met, model, hist
        print(f"[exp2] {s}^3 AUC={met['auc_roc']:.4f} AP={met['auc_pr']:.4f} F1={met['f1']:.4f} "
              f"bAcc={met['balanced_acc']:.4f} ECE={met['ece']:.4f} ({met['train_min']} min)", flush=True)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.4))
    for s, h in hists.items():
        axs[0].plot([x["epoch"] for x in h], [x["loss"] for x in h], label=f"{s}^3")
        axs[1].plot([x["epoch"] for x in h], [x["auc_roc"] for x in h], label=f"{s}^3")
    axs[0].set_title("Val loss"); axs[0].set_xlabel("epoch")
    axs[1].set_title("Val AUC-ROC"); axs[1].set_xlabel("epoch")
    for a in axs:
        a.legend(); a.grid(alpha=0.3)
    p2 = os.path.join(FIG_DIR, "exp2_learning_curves.png")
    fig.tight_layout(); fig.savefig(p2, dpi=160, bbox_inches="tight"); plt.close(fig)
    print("wrote", p2)
    if RUN_WANDB:
        import wandb
        for s, met in exp2.items():
            run.log({f"exp2/{s}/{k}": v for k, v in met.items()}, step=2)
        run.log({"exp2/img": wandb.Image(p2)}, step=2)


[exp2] train 200^3 ...


/content/glaucoma-thesis/scripts/compare_resolutions.py:128: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  losses.append(float(loss))


    epoch 1/30 (74s) auc=0.4819 loss=0.7052
    epoch 2/30 (146s) auc=0.6429 loss=0.6919
    epoch 3/30 (217s) auc=0.8074 loss=0.6282
    epoch 4/30 (289s) auc=0.8257 loss=0.5688
    epoch 5/30 (361s) auc=0.8358 loss=0.5568
    epoch 6/30 (433s) auc=0.8378 loss=0.5392
    epoch 7/30 (505s) auc=0.8436 loss=0.5305
    epoch 8/30 (577s) auc=0.8394 loss=0.5221
    epoch 9/30 (648s) auc=0.8417 loss=0.5167
    epoch 10/30 (721s) auc=0.8422 loss=0.5105
    epoch 11/30 (792s) auc=0.8334 loss=0.4977
    epoch 12/30 (864s) auc=0.8413 loss=0.4853
    epoch 13/30 (936s) auc=0.8463 loss=0.4743
    epoch 14/30 (1008s) auc=0.8465 loss=0.4612
    epoch 15/30 (1080s) auc=0.8474 loss=0.4403
    epoch 16/30 (1152s) auc=0.8326 loss=0.4194
    epoch 17/30 (1224s) auc=0.8422 loss=0.4000
    epoch 18/30 (1296s) auc=0.8449 loss=0.3631
    epoch 19/30 (1369s) auc=0.8417 loss=0.3318
    epoch 20/30 (1441s) auc=0.8235 loss=0.2918
    epoch 21/30 (1513s) auc=0.8366 loss=0.2519
    epoch 22/30 (1586s) auc=0.8284 l

## 7. Thí nghiệm 3 — CKA + Silhouette vs baseline 200³

In [11]:
exp3 = {}
if RUN_EXP3 and models:
    from torch.utils.data import DataLoader
    from sklearn.manifold import TSNE
    feats, emb, sil = {}, {}, {}
    for s in SIZES:
        va = cr.VolumeDataset(DATA[s]["val_v"], DATA[s]["val_l"], n_max=VAL_N, seed=SEED + 1)
        dl = DataLoader(va, batch_size=BS, shuffle=False, num_workers=0)
        _, yv, f = cr.predict(models[s], dl, DEVICE)
        feats[s] = f
        emb[s] = TSNE(n_components=2, perplexity=min(30, len(f) - 1), init="pca", random_state=0).fit_transform(f)
        sil[s] = rs.silhouette_np(cr.pca_np(f, 16), yv)
    exp3 = {"sil": {str(s): float(sil[s]) for s in SIZES}}
    for res in RES_LIST:
        exp3[f"cka_{res}"] = float(rs.linear_cka(feats[BASELINE_RES], feats[res]))
        exp3[f"sil_drop_{res}"] = float((sil[BASELINE_RES] - sil[res]) / sil[BASELINE_RES]) if sil[BASELINE_RES] else float("nan")
        print(f"[exp3] CKA(200,{res})={exp3[f'cka_{res}']:.4f} | sil drop={exp3[f'sil_drop_{res}']*100:.2f}%")
    fig, axs = plt.subplots(1, len(SIZES), figsize=(4.2 * len(SIZES), 4.6))
    for ax, s in zip(np.atleast_1d(axs), SIZES):
        ax.scatter(emb[s][:, 0], emb[s][:, 1], c=yv, s=8, cmap="coolwarm", alpha=0.7)
        ax.set_title(f"{s}^3 | sil={sil[s]:.3f}"); ax.axis("off")
    p3 = os.path.join(FIG_DIR, "exp3_features_tsne.png")
    fig.tight_layout(); fig.savefig(p3, dpi=160, bbox_inches="tight"); plt.close(fig)
    print("wrote", p3)
    if RUN_WANDB:
        import wandb
        run.log({f"exp3/{k}": v for k, v in exp3.items() if isinstance(v, float)}, step=3)
        run.log({"exp3/img": wandb.Image(p3)}, step=3)


[exp3] CKA(200,128)=0.5966 | sil drop=1.16%
[exp3] CKA(200,96)=0.5188 | sil drop=-18.11%
wrote figures/resolution_3way/exp3_features_tsne.png


## 8. Thí nghiệm 4 — Batch-size Trade-off

So `200³ + BS nhỏ` (đã có ở Exp2) với `96³ + BS lớn` — kiểm tra batch lớn có bù được mất chi tiết không.

In [12]:
exp4 = {}
if RUN_EXP4 and BASELINE_RES in exp2:
    print(f"[exp4] train 96^3 bs={BS_LARGE} ...", flush=True)
    _, m96l, _, _, _ = cr.train_proxy(
        DATA[96]["train_v"], DATA[96]["train_l"], DATA[96]["val_v"], DATA[96]["val_l"],
        BS_LARGE, EPOCHS, DEVICE, SEED, width=WIDTH, n_train=TRAIN_N, n_val=VAL_N)
    exp4 = {f"200_bs{BS}": exp2[BASELINE_RES], f"128_bs{BS}": exp2[128], f"96_bs{BS_LARGE}": m96l}
    print(f"[exp4] AUC 200 bs{BS}={exp2[BASELINE_RES]['auc_roc']:.4f} | "
          f"128 bs{BS}={exp2[128]['auc_roc']:.4f} | 96 bs{BS_LARGE}={m96l['auc_roc']:.4f}", flush=True)
    if RUN_WANDB:
        run.log({"exp4/auc_200_smallbs": exp2[BASELINE_RES]["auc_roc"],
                 "exp4/auc_128_smallbs": exp2[128]["auc_roc"],
                 "exp4/auc_96_largebs": m96l["auc_roc"]}, step=4)


[exp4] train 96^3 bs=8 ...
    epoch 1/30 (7s) auc=0.5759 loss=0.7034
    epoch 2/30 (13s) auc=0.5814 loss=0.6915
    epoch 3/30 (20s) auc=0.6197 loss=0.6870
    epoch 4/30 (26s) auc=0.6103 loss=0.6789
    epoch 5/30 (33s) auc=0.7107 loss=0.6613
    epoch 6/30 (39s) auc=0.7805 loss=0.6057
    epoch 7/30 (46s) auc=0.7975 loss=0.5589
    epoch 8/30 (53s) auc=0.8087 loss=0.5148
    epoch 9/30 (59s) auc=0.7993 loss=0.4847
    epoch 10/30 (66s) auc=0.8200 loss=0.4495
    epoch 11/30 (72s) auc=0.8012 loss=0.4027
    epoch 12/30 (79s) auc=0.8026 loss=0.3377
    epoch 13/30 (86s) auc=0.7832 loss=0.2664
    epoch 14/30 (92s) auc=0.7784 loss=0.1836
    epoch 15/30 (99s) auc=0.7889 loss=0.1156
    epoch 16/30 (105s) auc=0.7828 loss=0.0596
    epoch 17/30 (112s) auc=0.7806 loss=0.0246
    epoch 18/30 (119s) auc=0.7868 loss=0.0069
    epoch 19/30 (125s) auc=0.7886 loss=0.0034
    epoch 20/30 (132s) auc=0.7897 loss=0.0024
    epoch 21/30 (139s) auc=0.7908 loss=0.0019
    epoch 22/30 (145s) auc=0.791

## 9. Kết luận 3 bản + lưu TẤT CẢ vào Drive

Chọn **bản nhỏ nhất** đạt ngưỡng so với 200³ (tối ưu cho sweep: nhanh mà không mất chất lượng).
Nếu cả 96 và 128 đều fail → giữ 200³.

In [13]:
import csv as _csv

base_auc = exp2[BASELINE_RES]["auc_roc"] if BASELINE_RES in exp2 else float("nan")
checks, verdict = {}, {}
for res in RES_LIST:
    c1 = bool(exp1.get(res, {}).get("ssim", 0) > 0.85 and exp1.get(res, {}).get("psnr", 0) > 30.0)
    c2 = bool(BASELINE_RES in exp2 and res in exp2 and (base_auc - exp2[res]["auc_roc"]) < 0.015)
    c3 = bool(exp3.get(f"cka_{res}", 0) > 0.85 and exp3.get(f"sil_drop_{res}", 1) < 0.10)
    c4 = bool(f"96_bs{BS_LARGE}" in exp4 and res == 96 and
              exp4[f"96_bs{BS_LARGE}"]["auc_roc"] >= exp4[f"200_bs{BS}"]["auc_roc"] - 0.005) if res == 96 else True
    checks[str(res)] = {"exp1_vs200": c1, "exp2_auc_drop": c2, "exp3_cka_sil": c3, "exp4_batch": c4}
    verdict[str(res)] = all([c1, c2, c3, c4])
    print(f"[verdict] {res}^3: {checks[str(res)]} -> {'OK' if verdict[str(res)] else 'FAIL'}")

best = BASELINE_RES
for res in RES_LIST:
    if verdict[str(res)]:
        best = res
        break
report = {"baseline": BASELINE_RES, "sizes": list(SIZES), "config": {"epochs": EPOCHS, "bs": BS,
          "bs_large": BS_LARGE, "train_n": TRAIN_N, "val_n": VAL_N},
          "exp1": exp1, "exp2": exp2, "exp3": exp3, "exp4": exp4,
          "checks": checks, "verdict": verdict, "best_for_sweep": f"{best}^3"}
print(f"=> PHU HOP NHAT CHO SWEEP: {best}^3")

report_path = os.path.join(FIG_DIR, "resolution_3way_report.json")
with open(report_path, "w") as fh:
    json.dump(report, fh, indent=2, default=float)

csv_path = os.path.join(FIG_DIR, "resolution_3way_metrics.csv")
with open(csv_path, "w", newline="") as fh:
    w = _csv.writer(fh)
    w.writerow(["size", "auc_roc", "auc_pr", "f1", "balanced_acc", "ece", "train_min"])
    for s in SIZES:
        if s in exp2:
            x = exp2[s]
            w.writerow([f"{s}^3", x["auc_roc"], x["auc_pr"], x["f1"], x["balanced_acc"], x["ece"], x["train_min"]])

summary = ["# Kết quả so sánh 96^3 vs 128^3 vs 200^3 (Harvard-GF)", "",
           f"- Cấu hình: {EPOCHS} epochs, bs {BS}/{BS_LARGE}, train_n {TRAIN_N or 'all'}, val_n {VAL_N or 'all'}", "",
           "## Exp2 (proxy model)", "| Size | AUC-ROC | AUC-PR | F1 | Balanced Acc | ECE |", "|---|---|---|---|---|---|"]
for s in SIZES:
    if s in exp2:
        x = exp2[s]
        summary.append(f"| {s}^3 | {x['auc_roc']:.4f} | {x['auc_pr']:.4f} | {x['f1']:.4f} | {x['balanced_acc']:.4f} | {x['ece']:.4f} |")
summary += ["", "## Exp1 (vs 200^3)", "| Size | PSNR | SSIM |", "|---|---|---|"]
for res in RES_LIST:
    if res in exp1:
        summary.append(f"| {res}^3 | {exp1[res]['psnr']:.2f} | {exp1[res]['ssim']:.4f} |")
summary += ["", "## Exp3 (vs 200^3)", "| Size | CKA | Sil-drop |", "|---|---|---|"]
for res in RES_LIST:
    if f"cka_{res}" in exp3:
        summary.append(f"| {res}^3 | {exp3[f'cka_{res}']:.4f} | {exp3[f'sil_drop_{res}']*100:.2f}% |")
summary += ["", f"**Phù hợp nhất cho sweep: {best}^3**", "", "## Ghi chú (điền tay)", "- ", ""]
summary_path = os.path.join(FIG_DIR, "RESULT_96_128_200.md")
with open(summary_path, "w", encoding="utf-8") as fh:
    fh.write("\n".join(summary))
print("wrote", report_path, "|", csv_path, "|", summary_path)

if RUN_WANDB:
    run.summary.update({"best_for_sweep": f"{best}^3"})
    for res, v in verdict.items():
        run.summary.update({f"verdict/{res}": bool(v)})

if DRIVE_READY:
    import shutil
    os.makedirs(DRIVE_FIG, exist_ok=True)
    os.makedirs(DRIVE_MODEL, exist_ok=True)
    for f in os.listdir(FIG_DIR):
        shutil.copy2(os.path.join(FIG_DIR, f), os.path.join(DRIVE_FIG, f))
    for s, model in models.items():
        torch.save({"state_dict": model.state_dict(), "size": s}, os.path.join(DRIVE_MODEL, f"proxy_{s}.pt"))
    print("[drive] figures ->", DRIVE_FIG)
    print("[drive] models  ->", DRIVE_MODEL)
else:
    print("[drive] SKIP")

if RUN_WANDB:
    run.finish()
print("done")


[verdict] 128^3: {'exp1_vs200': False, 'exp2_auc_drop': True, 'exp3_cka_sil': False, 'exp4_batch': True} -> FAIL
[verdict] 96^3: {'exp1_vs200': False, 'exp2_auc_drop': True, 'exp3_cka_sil': False, 'exp4_batch': False} -> FAIL
=> PHU HOP NHAT CHO SWEEP: 200^3
wrote figures/resolution_3way/resolution_3way_report.json | figures/resolution_3way/resolution_3way_metrics.csv | figures/resolution_3way/RESULT_96_128_200.md
[drive] figures -> /content/drive/MyDrive/MasterBKDN/Thesis/resolution_3way_96_128_200_figures
[drive] models  -> /content/drive/MyDrive/MasterBKDN/Thesis/resolution_3way_96_128_200


exp1/128/psnr,▁
exp1/128/ssim,▁
exp1/96/psnr,▁
exp1/96/ssim,▁
exp2/128/auc_pr,▁
exp2/128/auc_roc,▁
exp2/128/balanced_acc,▁
exp2/128/ece,▁
exp2/128/f1,▁
exp2/128/precision,▁
+25,...


done


## 10. Ghi chú kết quả (điền tay)

Bản phù hợp nhất cho sweep: `____`³

Lý do / nhận xét:

-

Kết quả lưu tại Drive:
- `resolution_3way_96_128_200_figures/RESULT_96_128_200.md`
- `resolution_3way_96_128_200_figures/resolution_3way_report.json`
- `resolution_3way_96_128_200_figures/resolution_3way_metrics.csv`
- `resolution_3way_96_128_200/proxy_96.pt`, `proxy_128.pt`, `proxy_200.pt`
